In [1]:
import pathlib

import yaml
# import scipy.signal
import pandas as pd
# import numpy as np

import jax.random as jr
# from jaxtyping import Array, Float

# import torch
from torch.utils.data import DataLoader

import altair as alt
import plotly.express as px
import plotly.graph_objects as go

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core
import trapallada.plot

# import ipywidgets as widgets

So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
data_directory = pathlib.Path.cwd().parent / 'data'
assert data_directory.exists()
data_directory

PosixPath('/home/manu/Sync/git/trapallada/data')

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = '4_classes_modulated'
# base_filename = '20_600'
base_filename = '63_225'

signals_file = data_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

# metadata_file = data_directory / f'{base_filename}_labels.parquet'
metadata_file = data_directory / f'{base_filename}_metadata.parquet'
assert metadata_file.exists()

# signals_file, metadata_file

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# len(ds)

trapallada.data - WARNING - dismissed 198 constant signals (that cannot be normalized); 14885 left


The directory containing the actual results

In [6]:
# results_directory = pathlib.Path.cwd()
results_directory = pathlib.Path.cwd() / 'results'
# results_directory = pathlib.Path('/tmp')
assert results_directory.exists()
results_directory

PosixPath('/home/manu/Sync/git/trapallada/experiments/results')

In [7]:
list(results_directory.glob('*.eqx'))

[PosixPath('/home/manu/Sync/git/trapallada/experiments/results/good.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/3.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/4v3.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/real_z_20_4000_epochs.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/smaller.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/1.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/real_z_50_4000_epochs.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/5.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/4v2.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/4.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/2.eqx')]

*Base* file name for the results (trained model)

In [8]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_50_300_epochs' # <-------------------------------------
# input_files_stem = results_directory / 'new' / 'z_2_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_3_300_epochs' 
# input_files_stem = results_directory / 'new' / 'real_z_50_300_epochs'
# ---
# input_files_stem = results_directory / 'real_z_50_4000_epochs'
# input_files_stem = results_directory / 'real_z_20_4000_epochs'
# input_files_stem = results_directory / 'smaller'
# input_files_stem = results_directory / '1'
# input_files_stem = results_directory / '2' # <---- good
# input_files_stem = results_directory / '3'
# input_files_stem = results_directory / '4' # <---- good
input_files_stem = results_directory / '4v2' # <---- good
# input_files_stem = results_directory / '5' # <---- overfitting


In [9]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [10]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [11]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [12]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
# parameters
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
encoder:
  input_size:
  - 1
  - 225
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 50 
  kernel_sizes:
  - 64
  - 32
  - 16
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 20000
  train_test_split:
    test_size: 0.2


In [13]:
results_df = pd.read_parquet(results_file)
results_df

training                            validation                \
              loss loglikelihood         kl         loss loglikelihood   
0      1779.533023  -1761.267838  18.265183  1726.398649  -1708.381793   
1      1715.127553  -1697.502489  17.625066  1710.773578  -1693.449071   
2      1707.027618  -1690.194488  16.833127  1705.656494  -1689.213145   
3      1703.116683  -1687.297869  15.818812  1702.444763  -1687.175557   
4      1699.997328  -1685.545554  14.451773  1698.635984  -1684.982914   
...            ...           ...        ...          ...           ...   
14004  1140.611324  -1045.092710  95.518614  1186.310117  -1091.480477   
14005  1140.528665  -1044.902207  95.626458  1186.897791  -1091.605887   
14006  1140.500030  -1044.916379  95.583653  1185.406786  -1090.091849   
14007  1139.740557  -1044.169761  95.570793  1186.889419  -1091.521527   
14008  1140.529511  -1044.952056  95.577456  1185.721333  -1090.693141   

                  
              kl  
0      18.016845  
1      17.324509  
2      16.443349  
3      15.269205  
4      13.653070  
...          ...  
14004  94.829637  
14005  95.291907  
14006  95.314943  
14007  95.367895  
14008  95.028192  

[14009 rows x 6 columns]

In [14]:
# next(results_df.iterrows())[1].to_dict()
# # {k:v for k,v in next(results_df.iterrows())[1].to_dict()}
# {'_'.join(k):v for k,v in next(results_df.iterrows())[1].to_dict().items()}

# Training curves

In [15]:
def metric_chart(df: pd.DataFrame, metric: str, n_rows: int = 4000):

    sub_df = df.loc[:n_rows,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [16]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [17]:
sum(trapallada.util.get_layers_n_parameters(model).values())

29789

Data loaders for both the training and validation sets

In [18]:
(train_dl, validation_dl), (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

A batch from the **validation** set

In [19]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl), next(iter_dl)
batch = next(iter_dl)
batch[1].shape

torch.Size([32, 1, 225])

What are the different labels, *if any* (useful to check whether every class is present)?

In [20]:
batch[2].unique()

tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan], dtype=torch.float64)

An element from the above batch is encoded, decoded, and a prediction plotted

In [21]:
# i = 0
# i = 1
i = 31

# indexes from the above batch
i_s, *_ = batch

def plot_prediction(ds, i):
    
    x = ds[i][1].numpy()
    x_pred, z_mean, z_sd = model(x, jr.PRNGKey(42))
    display(ds.metadata_df.loc[i])
    df = pd.DataFrame({'prediction': x_pred[0], 'actual': x.ravel()})
    n_rows = 2000
    
    return trapallada.plot.df(df.iloc[:n_rows])

# plot_prediction(ds, 25)
plot_prediction(ds, i_s[i].item())

start            2019-11-22 00:14:14.532324
end              2019-11-22 05:57:38.052324
tar file                     0853980901.tar
source      P0853980901PNS001SRCTSR8002.FTZ
Name: 169, dtype: object

alt.Chart(...)

## Javi's paper

Sources discussed in this paper are part of the above dataset.

In [22]:
javi_df = ds.metadata_df[ds.metadata_df['source'] == 'P0605950101PNS003SRCTSR8001.FTZ']
javi_df.tail()

,start,end,tar file,source
5914,2009-06-03 10:45:23.497454,2009-06-03 16:28:47.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5918,2009-06-03 12:02:02.497454,2009-06-03 17:45:26.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5922,2009-06-03 13:18:41.497454,2009-06-03 19:02:05.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5926,2009-06-03 14:35:20.497454,2009-06-03 20:18:44.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5930,2009-06-03 15:19:47.917454,2009-06-03 21:03:11.437454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ


In [23]:
(javi_df['end'] - javi_df['start']).tail()

5914   0 days 05:43:23.520000
5918   0 days 05:43:23.520000
5922   0 days 05:43:23.520000
5926   0 days 05:43:23.520000
5930   0 days 05:43:23.520000
dtype: timedelta64[ns]

In [24]:
n_signals_to_plot = 1
trapallada.plot.signals_array(ds[javi_df.tail(n_signals_to_plot).index][1].numpy().squeeze(axis=1))

alt.Chart(...)

In [25]:
i_javi = javi_df.tail(1).index.item()
plot_prediction(ds, i_javi)

start            2009-06-03 15:19:47.917454
end              2009-06-03 21:03:11.437454
tar file                     0605950101.tar
source      P0605950101PNS003SRCTSR8001.FTZ
Name: 5930, dtype: object

alt.Chart(...)

Was this signal seen during training?

In [26]:
i_javi in train_ds[:][0]
# i_javi in validation_ds[:][0]

True

# Latent space projection

(Out of the full dataset), the curves in the given `DataLoader` are encoded into the latent space (last columns in the resulting `DataFrame`)

In [27]:
fulldataset_dl = DataLoader(ds, batch_size=parameters['training']['batch_size'], shuffle=False)
proj_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder)
proj_df.loc[train_ds[:][0], 'set'] = 'training'
proj_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj_df

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2,set
0,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8001.FTZ,-0.590175,-1.923177,0.045313,1.285682,0.792428,-0.077937,...,0.296039,-1.093156,1.740326,1.017020,0.959759,-0.239907,0.045339,14.952848,-4.390003,training
1,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8002.FTZ,0.360476,0.047649,-0.752770,0.703445,-2.449885,1.135201,...,-0.815416,1.797720,0.108298,0.358391,-0.480961,-1.750277,-0.555723,16.586473,2.255387,training
2,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8003.FTZ,-0.910477,0.375480,-0.054875,0.070794,-0.047069,-1.744383,...,0.504260,0.199786,-0.480764,-1.230273,-1.726619,-0.016346,0.161111,-24.239195,-8.780774,training
3,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8004.FTZ,1.078361,1.413429,-0.974158,0.627543,1.119117,-1.141359,...,0.138536,-0.495018,2.010034,-0.125806,-0.690734,0.664101,0.166524,-10.187140,-32.391243,validation
4,2018-06-08 21:31:20.569583,2018-06-09 03:14:44.089583,0821871301.tar,P0821871301PNS003SRCTSR8001.FTZ,0.177224,1.045853,-0.102202,0.077235,-0.224060,-0.932703,...,0.292697,-2.231858,0.986130,-0.569833,-1.334947,-0.424121,-0.384886,-13.435054,-0.062588,training
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14880,2014-04-16 02:44:34.529122,2014-04-16 08:27:58.049122,0728960601.tar,P0728960601PNS003SRCTSR8004.FTZ,-2.508382,0.349323,-1.425077,1.135122,0.771209,-0.989898,...,-0.325588,-0.692310,2.109652,0.679399,0.260578,-1.743588,1.089542,-16.682997,20.875181,training
14881,2014-04-16 03:19:50.069122,2014-04-16 09:03:13.589122,0728960601.tar,P0728960601PNS003SRCTSR8001.FTZ,1.418517,0.533909,-0.074533,-1.235355,-0.078656,0.218199,...,-0.105585,-1.702446,-1.059487,-0.239322,-0.422585,-0.964196,-1.365199,9.190551,6.636133,training
14882,2014-04-16 03:19:50.069122,2014-04-16 09:03:13.589122,0728960601.tar,P0728960601PNS003SRCTSR8002.FTZ,-0.188453,1.435612,-0.155575,-0.327711,0.570619,-0.393245,...,-1.287012,1.287492,2.126611,0.771167,-0.653459,-0.293807,-0.324465,-15.788837,-0.557947,training
14883,2014-04-16 03:19:50.069122,2014-04-16 09:03:13.589122,0728960601.tar,P0728960601PNS003SRCTSR8003.FTZ,-1.005950,-1.164055,1.065143,-2.617605,0.486818,0.863034,...,0.306192,-0.776183,0.421269,1.140779,-1.087408,-1.714950,2.366468,-48.063038,-1.982899,validation


In [28]:
alt.Chart(proj_df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='tar file:N',
    fill='tar file:N',
    shape='set:N',
    tooltip=['source:N', 'tsne_1:Q', 'tsne_2:Q'],
    # tooltip=['tar file:N']
).properties(
    width=800, height=600
).interactive()

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

Embedded space is close to a standard Gaussian?

In [29]:
proj_df.filter(like='z_').agg(['mean', 'std'])

,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,z_9,z_10,...,z_41,z_42,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50
mean,-0.044402,0.014226,0.022470,-0.028120,0.010601,0.007473,-0.012044,0.005292,-0.045951,-0.041669,...,-0.023340,0.027564,0.012218,-0.013728,0.003083,0.006650,0.095860,-0.004431,-0.027529,0.018478
std,0.984183,0.955992,0.988545,0.943638,0.943660,0.961064,0.954736,0.962060,0.959505,0.952671,...,0.967353,0.959316,0.978840,0.971084,0.986016,0.985995,0.961959,0.983608,0.956659,0.977552


In [30]:
# valid_proj_df = trapallada.util.make_projections_df(ds, validation_dl, model.encoder)
# valid_proj_df.head()

Embedded space is close to a standard Gaussian?

In [31]:
# valid_proj_df.filter(like='z_').agg(['mean', 'std'])

In [32]:
# def plot_projections(df: pd.DataFrame, n_rows: int = 1_000):

#     chart = alt.Chart(df.sample(n=n_rows)).mark_point().encode(
#         x='tsne_1:Q',
#         y='tsne_2:Q',
#         color='tar file:N',
#         fill='tar file:N',
#         tooltip=['source:N', 'tsne_1:Q', 'tsne_2:Q']
#         # tooltip=['tar file:N']
#     ).properties(
#         width=800, height=600
#     ).interactive()

#     return chart

# plot_projections(valid_proj_df)

Projections also for the training set, which encompass...

In [33]:
# train_proj_df = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
# train_proj_df.head()

...Javi's paper

In [34]:
# train_proj_df.loc[i_javi, ['tsne_1', 'tsne_2']]
proj_df.loc[i_javi, ['tsne_1', 'tsne_2']]

tsne_1     6.782359
tsne_2   -48.958527
Name: 5930, dtype: object

In [35]:
# # alt.Chart(train_proj_df.sample(n=1_000)).mark_point().encode(
# #     x='tsne_1:Q',
# #     y='tsne_2:Q',
# #     color='tar file:N',
# #     fill='tar file:N',
# #     tooltip=['source:N', 'tsne_1:Q', 'tsne_2:Q']
# #     # tooltip=['tar file:N']
# # ).properties(
# #     width=800, height=600
# # ).interactive()

# plot_projections(train_proj_df)

The latent space is projected into a 3D space

In [36]:
proj3D_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder, n_projected_components=3)
proj3D_df.head()

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2,tsne_3
0,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8001.FTZ,-0.590175,-1.923177,0.045313,1.285682,0.792428,-0.077937,...,0.296039,-1.093156,1.740326,1.017020,0.959759,-0.239907,0.045339,-2.173821,27.838802,-24.989983
1,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8002.FTZ,0.360476,0.047649,-0.752770,0.703445,-2.449885,1.135201,...,-0.815416,1.797720,0.108298,0.358391,-0.480961,-1.750277,-0.555723,-17.869806,-18.770548,-6.695476
2,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8003.FTZ,-0.910477,0.375480,-0.054875,0.070794,-0.047069,-1.744383,...,0.504260,0.199786,-0.480764,-1.230273,-1.726619,-0.016346,0.161111,36.929493,-36.552032,28.534264
3,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNS003SRCTSR8004.FTZ,1.078361,1.413429,-0.974158,0.627543,1.119117,-1.141359,...,0.138536,-0.495018,2.010034,-0.125806,-0.690734,0.664101,0.166524,3.920672,40.821613,16.285057
4,2018-06-08 21:31:20.569583,2018-06-09 03:14:44.089583,0821871301.tar,P0821871301PNS003SRCTSR8001.FTZ,0.177224,1.045853,-0.102202,0.077235,-0.224060,-0.932703,...,0.292697,-2.231858,0.986130,-0.569833,-1.334947,-0.424121,-0.384886,37.420994,-49.041824,14.400215


In [37]:
proj3D_df.loc[i_javi].filter(like='tsne_')

tsne_1   -34.363281
tsne_2    -1.276066
tsne_3   -43.696644
Name: 5930, dtype: object

In [105]:
# fig = px.scatter_3d(proj3D_df, x='tsne_1', y='tsne_2', z='tsne_3', color='tar file', opacity=0.5, width=1_200, height=800)
fig = px.scatter_3d(proj3D_df.reset_index(), x='tsne_1', y='tsne_2', z='tsne_3', hover_data=['index'], color='tar file', opacity=0.5, width=1_200, height=800)
fig.add_trace(go.Scatter3d(
    x=[proj3D_df.loc[i_javi, 'tsne_1']], y=[proj3D_df.loc[i_javi, 'tsne_2']], z=[proj3D_df.loc[i_javi, 'tsne_3']], mode='markers', marker=dict(size=12, symbol='diamond', color='blue')
))
fig.show()

In [94]:
map(list, [2, 3])

In [95]:
dict(zip(['x', 'y'], [[e] for e in proj_df.loc[i_javi].filter(like='tsne_')]))

{'x': [6.782359], 'y': [-48.958527]}

In [98]:
# fig = px.scatter(proj_df, x='tsne_1', y='tsne_2', color='tar file', opacity=0.5, width=1_200, height=800)
fig = px.scatter(proj_df.reset_index(), x='tsne_1', y='tsne_2', color='tar file', hover_data=['index'], opacity=0.5, width=1_200, height=800)
# fig.add_trace(go.Scatter(
#     **dict(zip(['x', 'y'], [[e] for e in proj_df.loc[i_javi].filter(like='tsne_')])), mode='markers', marker=dict(size=20, symbol='star', color='blue')
# ))
fig.add_trace(go.Scatter(
    x=[proj_df.loc[i_javi, 'tsne_1']], y=[proj_df.loc[i_javi, 'tsne_2']], mode='markers', marker=dict(size=20, symbol='star', color='blue')
))
fig.show()

In [64]:
# fig.add_trace(go.Scatter(
#     x=[proj_df.loc[i_javi, 'tsne_1']], y=[proj_df.loc[i_javi, 'tsne_2']], mode='markers', marker=dict(size=20, symbol='star', color='blue')
# ))

In [115]:
# ds[2239]
# plot_prediction(ds, 2239)
# plot_prediction(ds, 12348)
# plot_prediction(ds, 10480)
# plot_prediction(ds, 5778)
# plot_prediction(ds, 3995)
# plot_prediction(ds, 7102)
plot_prediction(ds, 10190)

start            2012-10-02 14:49:40.626867
end              2012-10-02 20:33:04.146867
tar file                     0694640701.tar
source      P0694640701PNS003SRCTSR8003.FTZ
Name: 10190, dtype: object

alt.Chart(...)